# GinSign Grounder — Joint Model Evaluation

Loads the **single** BERT model trained jointly on predicate + argument shards and evaluates it on the **test** split with a breakdown:
- Overall metrics
- **Per domain** (`<search_and_rescue>`, `<warehouse>`, `<traffic_light>`) 
- **Per predicate** (tokens after `<predicates>`) 
- **Per constant** (tokens after `<const>`)

In [ ]:

# Install deps (safe to run multiple times)
import sys, subprocess, pkgutil
def _pip(pkg): 
    if pkgutil.find_loader(pkg.split("==")[0]) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
_pip("datasets")
_pip("transformers")
_pip("scikit-learn")
_pip("pandas")
_pip("accelerate")


In [ ]:

from pathlib import Path
from typing import List, Dict, Any, Tuple, Iterable
import os, json, math, numpy as np, pandas as pd

import torch
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# --- Config ---
DATA_ROOT = Path("libero_grounding_data")            # path produced by build_grounding_data.py
MODEL_DIR = OUT_DIR = Path("outputs_joint")             # folder where the single joint model was saved
BATCH_SIZE = 32
MAX_PREFIX_SHARD = 20                         # must match data builder
THRESH = 0.5

assert MODEL_DIR.exists(), f"Model dir not found: {MODEL_DIR}"


In [ ]:

# -------------------------------
# Data loading
# -------------------------------
def _read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        print(f"[warn] File not found: {path}")
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows
SUITES = ["libero_10", "libero_90", "libero_object", "libero_goal", "libero_spatial"]
LIBERO_DATA_DIR = Path("libero_grounding_data")  # Output of convert_libero_dataset.py

def load_libero_data(data_dir: Path, suites: List[str]) -> List[Dict[str, Any]]:
    """Load LIBERO grounding data from JSONL files."""
    all_rows = []
    for suite in suites:
        path = data_dir / f"{suite}_grounding.jsonl"
        rows = _read_jsonl(path)
        print(f"  {suite}: {len(rows)} entries")
        all_rows.extend(rows)
    return all_rows

print(f"Loading LIBERO data from: {LIBERO_DATA_DIR}")
test_rows = load_libero_data(LIBERO_DATA_DIR, SUITES)
print(f"\nTotal: {len(test_rows)} entries")

if test_rows:
    print(f"Sample entry keys: {list(test_rows[0].keys())}")

In [ ]:

# -------------------------------
# Derive helper fields per row
# -------------------------------
def row_domain(row: Dict[str, Any]) -> str:
    # domain is the first token of the prefix
    return row["prefix"][0] if row.get("prefix") else "<?>"

def row_task(row: Dict[str, Any]) -> str:
    pf = row.get("prefix", [])
    if "<predicates>" in pf:
        return "predicate"
    if "<const>" in pf:
        return "argument"
    return "unknown"

def marker_index(prefix: List[str], marker: str) -> int:
    try:
        return prefix.index(marker)
    except ValueError:
        return -1

# attach task and domain for grouping later
for r in test_rows:
    r["_domain"] = row_domain(r)
    r["_task"] = row_task(r)



In [ ]:

# -------------------------------
# Tokenization on the fly in a collator so we can keep original metadata
# -------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

class GrounderDataset(torch.utils.data.Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx): return self.rows[idx]

def collate(batch: List[Dict[str, Any]]):
    sents = [" ".join(b["sentence"]) if isinstance(b["sentence"], list) else str(b["sentence"]) for b in batch]
    prefixes = [" ".join(b["prefix"]) if isinstance(b["prefix"], list) else str(b["prefix"]) for b in batch]
    enc = tokenizer(sents, prefixes, padding=True, truncation=True, max_length=512, return_tensors="pt")
    labels = [b["prefix_target"] for b in batch]
    # pad labels to MAX_PREFIX_SHARD
    labels = [ (l + [0]*max(0, MAX_PREFIX_SHARD-len(l)))[:MAX_PREFIX_SHARD] for l in labels ]
    enc["labels"] = torch.tensor(labels, dtype=torch.float32)
    # also pass through original objects for analysis
    enc["meta"] = batch
    return enc



In [ ]:

from torch.utils.data import DataLoader
dl = DataLoader(GrounderDataset(test_rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

# Run prediction
all_probs, all_preds, all_labels, all_meta = [], [], [], []
with torch.no_grad():
    for enc in dl:
        labels = enc.pop("labels").to(device)
        meta = enc.pop("meta")
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        logits = out.logits
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).to(torch.int)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_meta.extend(meta)

import numpy as np
y_prob = np.vstack(all_probs)
y_pred = np.vstack(all_preds).astype(int)
y_true = np.vstack(all_labels).astype(int)
len(all_meta), y_true.shape, y_pred.shape


In [ ]:

# -------------------------------
# Metrics helpers
# -------------------------------
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import pandas as pd

def overall_metrics(y_true, y_pred):
    pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
    pM, rM, fM, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    return {"accuracy": acc, "f1_micro": fm, "f1_macro": fM, "precision_micro": pm, "recall_micro": rm,
            "precision_macro": pM, "recall_macro": rM}

overall = overall_metrics(y_true, y_pred)
pd.Series(overall)


In [ ]:

# -------------------------------
# Per-domain metrics
# -------------------------------
def mask_rows(cond_iter):
    idxs = np.nonzero(np.array(list(cond_iter)))[0]
    return y_true[idxs], y_pred[idxs]

domains = sorted(set(r["_domain"] for r in all_meta))
rows = []
for d in domains:
    y_t_d, y_p_d = mask_rows(r["_domain"] == d for r in all_meta)
    m = overall_metrics(y_t_d, y_p_d)
    m["domain"] = d
    m["n_rows"] = len(y_t_d)
    rows.append(m)

df_domain = pd.DataFrame(rows).set_index("domain").sort_values("f1_micro", ascending=False)
df_domain


In [ ]:

# -------------------------------
# Per-predicate metrics (predicate shards only, tail after "<predicates>")
# -------------------------------
from collections import defaultdict

def per_item_metrics(meta_rows, y_true, y_pred, tail_marker: str) -> pd.DataFrame:
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int); sup = defaultdict(int)

    for r, yt, yp in zip(meta_rows, y_true, y_pred):
        prefix = r["prefix"]
        if tail_marker not in prefix:
            continue
        m = prefix.index(tail_marker)
        for j in range(m+1, len(prefix)):
            tok = prefix[j]
            yj = int(yt[j]) if j < len(yt) else 0
            pj = int(yp[j]) if j < len(yp) else 0
            sup[tok] += yj
            if yj == 1 and pj == 1: tp[tok] += 1
            elif yj == 0 and pj == 1: fp[tok] += 1
            elif yj == 1 and pj == 0: fn[tok] += 1

    rows = []
    for tok in sorted(set(list(tp) + list(fp) + list(fn) + list(sup))):
        t, f_p, f_n, s = tp[tok], fp[tok], fn[tok], sup[tok]
        prec = t / (t + f_p) if (t + f_p) > 0 else 0.0
        rec  = t / (t + f_n) if (t + f_n) > 0 else 0.0
        f1   = (2*prec*rec / (prec+rec)) if (prec+rec) > 0 else 0.0
        rows.append({"token": tok, "support": int(s), "tp": int(t), "fp": int(f_p), "fn": int(f_n),
                     "precision": prec, "recall": rec, "f1": f1})
    df = pd.DataFrame(rows).sort_values(["f1","support"], ascending=[False, False])
    return df

pred_mask = [("<predicates>" in r["prefix"]) for r in all_meta]
meta_pred = [r for r, m in zip(all_meta, pred_mask) if m]
y_true_pred = y_true[np.array(pred_mask)]
y_pred_pred = y_pred[np.array(pred_mask)]

df_predicate = per_item_metrics(meta_pred, y_true_pred, y_pred_pred, "<predicates>")
df_predicate.head(20)


In [ ]:

# -------------------------------
# Per-constant metrics (argument shards only, tail after "<const>")
# -------------------------------
arg_mask = [("<const>" in r["prefix"]) for r in all_meta]
meta_arg = [r for r, m in zip(all_meta, arg_mask) if m]
y_true_arg = y_true[np.array(arg_mask)]
y_pred_arg = y_pred[np.array(arg_mask)]

df_constant = per_item_metrics(meta_arg, y_true_arg, y_pred_arg, "<const>")
df_constant.head(20)


In [ ]:

# Save artifacts
out_dir = MODEL_DIR / "eval_breakdown"
out_dir.mkdir(parents=True, exist_ok=True)
df_domain.to_csv(out_dir / "per_domain.csv", index=True)
df_predicate.to_csv(out_dir / "per_predicate.csv", index=False)
df_constant.to_csv(out_dir / "per_constant.csv", index=False)

print("Saved:")
print(out_dir / "per_domain.csv")
print(out_dir / "per_predicate.csv")
print(out_dir / "per_constant.csv")


In [ ]:

# ============================================
# SAVE RAW PREDICTIONS into copies of the original test JSONLs
# ============================================
from pathlib import Path
import json, numpy as np, pandas as pd

def _load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                rows.append(json.loads(s))
    return rows

def _write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def _first_present(d, names):
    for n in names:
        if n in d and d[n] is not None:
            return d[n]
    return None

def _norm_list(x):
    # convert torch/tensor/np to plain list
    try:
        import torch
        if isinstance(x, torch.Tensor):
            x = x.detach().cpu().numpy()
    except Exception:
        pass
    if isinstance(x, np.ndarray):
        return x.tolist()
    return list(x)


def _coerce_per_example(name, arr, n):
    """
    Try to coerce `arr` into an n-length per-example sequence.
    - If arr has length n, return as list
    - If arr is scalar/length-1, broadcast
    - If arr is an (n, K) array, keep per-example rows (as lists)
    - Else: drop (return None) with a warning
    """
    if arr is None:
        return None
    try:
        import torch, numpy as np
        if isinstance(arr, torch.Tensor):
            arr = arr.detach().cpu().numpy()
        a = np.asarray(arr, dtype=object)
    except Exception:
        try:
            a = list(arr)
        except Exception:
            print(f"[warn] '{name}' not list-like; dropping")
            return None

    # Exact match
    try:
        L = len(a)
    except Exception:
        L = None

    if L == n:
        return a.tolist() if hasattr(a, 'tolist') else list(a)

    # Broadcast length-1
    if L == 1:
        return [a[0]] * n

    # (n, K) shape (e.g., per-example class-prob rows)
    try:
        import numpy as np
        a2 = np.asarray(a)
        if getattr(a2, 'ndim', 1) == 2 and a2.shape[0] == n:
            return a2.tolist()
    except Exception:
        pass

    print(f"[warn] Dropping '{name}' because length={L} != n={n} and not broadcastable or (n,K)-shaped.")
    return None

# ---- Collect predictions and metadata from the current environment ----
_loc = locals()

# Required-ish: metadata per example (needed to group back by file)
all_meta = _loc.get("all_meta", None)
if all_meta is None:
    raise RuntimeError("Expected `all_meta` to exist (list of per-example dicts). Run your evaluation cell first.")

# Try to get an id per example (optional but helpful)
ids = _first_present(_loc, ["ids","example_ids","idxs","idx","id_list"])
if ids is not None:
    ids = _norm_list(ids)

# Try to get core predictions
y_pred = _first_present(_loc, ["y_pred","preds","predictions","yhat"])
if y_pred is not None:
    y_pred = _norm_list(y_pred)

# Optional detailed predictions
y_pred_action = _first_present(_loc, ["y_pred_action","pred_actions","pred_action"])
if y_pred_action is not None:
    y_pred_action = _norm_list(y_pred_action)

y_pred_args = _first_present(_loc, ["y_pred_args","pred_arguments","pred_args"])
if y_pred_args is not None:
    y_pred_args = _norm_list(y_pred_args)

# Optional scores/probs
scores = _first_present(_loc, ["scores","all_scores","score"])
if scores is not None:
    scores = _norm_list(scores)

probs = _first_present(_loc, ["probs","all_probs","probabilities","prob"])
if probs is not None:
    probs = _norm_list(probs)

conf = _first_present(_loc, ["confidences","conf"])
if conf is not None:
    conf = _norm_list(conf)

# Correctness flags (if computed)
correct = _first_present(_loc, ["correct","is_correct","correct_overall"])
if correct is not None:
    correct = _norm_list(correct)
correct_action = _first_present(_loc, ["correct_action","action_correct"])
if correct_action is not None:
    correct_action = _norm_list(correct_action)
correct_args = _first_present(_loc, ["correct_args","args_correct"])
if correct_args is not None:
    correct_args = _norm_list(correct_args)

n = len(all_meta)
def _len(x): 
    try: return len(x)
    except: return None

# Coerce arrays to per-example length n (or drop with warning)
ids = _coerce_per_example('ids', ids, n)
y_pred = _coerce_per_example('y_pred', y_pred, n)
y_pred_action = _coerce_per_example('y_pred_action', y_pred_action, n)
y_pred_args = _coerce_per_example('y_pred_args', y_pred_args, n)
scores = _coerce_per_example('scores', scores, n)
probs = _coerce_per_example('probs', probs, n)
conf = _coerce_per_example('conf', conf, n)
correct = _coerce_per_example('correct', correct, n)
correct_action = _coerce_per_example('correct_action', correct_action, n)
correct_args = _coerce_per_example('correct_args', correct_args, n)

# ---- Determine how to group examples back to their source file ----

# Look for a per-example key that points to the originating JSONL path or stem.
file_key_candidates = ["file","jsonl_path","source_path","src","dataset_file","jf","stem"]
first_meta = all_meta[0] if n>0 else {}
file_key = None
for k in file_key_candidates:
    if k in first_meta:
        file_key = k
        break

# Build group index
group_index = {}
for i, m in enumerate(all_meta):
    key = None
    if file_key is not None:
        key = m.get(file_key)
    # Fallback: if we can't find a file per example, put everything in one 'combined' group
    if key is None:
        key = "__combined__"
    group_index.setdefault(key, []).append(i)

# ---- Helper to assemble per-example prediction dict ----
def _example_pred(i):
    p = {}
    if ids is not None:            p["id"] = ids[i]
    if y_pred is not None:         p["pred"] = y_pred[i]
    if y_pred_action is not None:  p["pred_action"] = y_pred_action[i]
    if y_pred_args is not None:    p["pred_args"] = y_pred_args[i]
    if scores is not None:         p["score"] = scores[i]
    if probs is not None:          p["prob"] = probs[i]
    if conf is not None:           p["conf"] = conf[i]
    if correct is not None:        p["correct"] = bool(correct[i])
    if correct_action is not None: p["correct_action"] = bool(correct_action[i])
    if correct_args is not None:   p["correct_args"] = bool(correct_args[i])
    return p

# ---- Write outputs per group ----
# Need OUT_DIR and TEST_DIR to be defined earlier in your notebook (they already are in your eval code).
try:
    OUT_DIR  # noqa
except NameError:
    raise RuntimeError("OUT_DIR is not defined. Define it earlier (same as you use for metrics).")
try:
    TEST_DIR  # noqa
except NameError:
    # TEST_DIR may not be needed if meta has absolute file paths; still warn.
    print("[warn] TEST_DIR not defined; relying on file paths inside all_meta (if provided).")

OUT_DIR.mkdir(parents=True, exist_ok=True)

for group_key, idxs in group_index.items():
    # Resolve the original JSONL path
    if group_key == "__combined__":
        # If we don't have per-file info, dump one combined JSONL
        stem = "combined"
        src_path = None
        original_rows = [all_meta[i].get("_original_row", None) for i in idxs]
        original_rows = None  # we generally don't have original rows embedded; load from file instead if possible
    else:
        # Try as a Path; if it's just a stem, pair with TEST_DIR
        candidate = Path(str(group_key))
        if candidate.suffix.lower() == ".jsonl" and candidate.exists():
            src_path = candidate
            stem = candidate.stem
        else:
            # assume it's a stem, use TEST_DIR/<stem>.jsonl
            try:
                TEST_DIR  # noqa
                src_path = Path(TEST_DIR) / f"{candidate.stem}.jsonl"
                stem = candidate.stem
            except NameError:
                src_path = None
                stem = candidate.stem

    # Load original rows if we have a source path
    if src_path is not None and src_path.exists():
        rows = _load_jsonl(src_path)
    else:
        # As a last resort, synthesize rows from meta (still preserving predictions)
        rows = [dict(all_meta[i]) for i in idxs]

    # Build a fast id index into `rows` if possible
    candidate_id_keys = ["id", "example_id", "idx"]
    row_ids = None
    id_key = None
    if rows and isinstance(rows[0], dict):
        present = set().union(*[set(r.keys()) for r in rows])
        for k in candidate_id_keys:
            if k in present:
                id_key = k
                row_ids = {r.get(k): j for j, r in enumerate(rows)}
                break

    # Merge predictions into rows
    for k, i in enumerate(idxs):
        pred = _example_pred(i)
        if id_key and ("id" in pred) and (pred["id"] in row_ids):
            j = row_ids[pred["id"]]
            out = dict(rows[j])
            out["predictions"] = pred
            rows[j] = out
        else:
            # Fallback: append in order if lengths align
            if len(rows) == len(idxs):
                j = k
                out = dict(rows[j]) if isinstance(rows[j], dict) else {"_row": rows[j]}
                out["predictions"] = pred
                rows[j] = out
            else:
                # If we can't align, just attach predictions field to a meta copy
                rmeta = dict(all_meta[i])
                rmeta["predictions"] = pred
                rows.append(rmeta)

    # Write
    out_path = Path(OUT_DIR) / f"{stem}_test_with_predictions.jsonl"
    _write_jsonl(out_path, rows)
    print(f"[ok] Wrote: {out_path}")


In [ ]:
# === Full-group accuracy by domain from combined_test_with_predictions.jsonl ===
from pathlib import Path
import json, numpy as np, pandas as pd

# Change this if your file is elsewhere
COMBINED_PATH = Path("outputs_joint/combined_test_with_predictions.jsonl")

def load_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                rows.append(json.loads(s))
            except json.JSONDecodeError:
                try:
                    rows.append(json.loads(s.rstrip(", ")))
                except Exception:
                    continue
    return rows

def normalize_domain(prefix, row):
    dom = row.get("_domain")
    if dom is None and isinstance(prefix, list) and prefix:
        dom = prefix[0]
    if isinstance(dom, str) and dom.startswith("<") and dom.endswith(">"):
        dom = dom[1:-1]
    return dom

def target_index_from_row(row):
    pt = row.get("prefix_target")
    if isinstance(pt, list) and len(pt) > 0:
        try:
            arr = np.asarray(pt, dtype=float)
            if arr.ndim == 1 and arr.size > 0:
                return int(arr.argmax())
        except Exception:
            pass
    prefix = row.get("prefix", [])
    if not isinstance(prefix, list):
        return None
    try:
        pred_marker_idx = prefix.index("<predicates>")
        pred_candidates = prefix[pred_marker_idx + 1 :]
    except ValueError:
        pred_candidates = []
    tgt_tokens = row.get("target")
    if isinstance(tgt_tokens, list) and len(tgt_tokens) >= 1:
        tgt = tgt_tokens[0]
        if isinstance(tgt, str) and pred_candidates:
            try:
                return pred_candidates.index(tgt)
            except ValueError:
                return None
    return None

def pred_argmax_from_row(row):
    pred = row.get("predictions", {}).get("pred")
    if pred is None:
        return None
    try:
        arr = np.asarray(pred, dtype=float)
        if arr.ndim == 1 and arr.size > 0:
            return int(arr.argmax())
    except Exception:
        pass
    return None

def compute_correct(row):
    ti = target_index_from_row(row)
    pi = pred_argmax_from_row(row)
    if ti is None or pi is None:
        return None
    return int(ti == pi)

# --- Read & score
rows = load_jsonl(COMBINED_PATH)
if not rows:
    raise FileNotFoundError(f"No rows read from {COMBINED_PATH} (set COMBINED_PATH if needed).")

enriched = []
for r in rows:
    dom = normalize_domain(r.get("prefix", []), r)
    corr = compute_correct(r)
    if dom is None or corr is None:
        continue
    enriched.append({"domain": dom, "id": r.get("id"), "prop_id": r.get("prop_id"), "correct": corr})

if not enriched:
    raise RuntimeError("No scorable rows found (missing targets/predictions).")

df = pd.DataFrame(enriched)

# Group (domain, id) -> full-correct if ALL its prop rows are correct
grp = df.groupby(["domain", "id"])["correct"]
full_correct = (grp.mean() == 1.0).reset_index(name="full_correct")

per_dom = full_correct.groupby("domain")["full_correct"].agg(
    total_groups="count",
    full_groups="sum"
).reset_index()

per_dom["percent_full_correct"] = 100.0 * per_dom["full_groups"] / per_dom["total_groups"]
per_dom = per_dom.sort_values("domain").reset_index(drop=True)

print("Full-group (all props correct) accuracy by domain:")
for _, row in per_dom.iterrows():
    print(f"  {row['domain']}: {row['percent_full_correct']:.2f}%  "
          f"({int(row['full_groups'])}/{int(row['total_groups'])} groups)")

# Optional: display as a small table if you're in a notebook
try:
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user("Full-group accuracy per domain", per_dom)
except Exception:
    pass
